# CO₂ Emissions Prediction: Missing-Value Strategies and Model Evaluation

This notebook studies how different missing-value handling strategies affect the prediction of annual CO₂ emissions. The analysis combines emissions data with socioeconomic and energy-related variables, compares four imputation strategies, and evaluates four predictive models using time-based rolling-window validation.

The project originated as my graduation research in Information Systems and Technologies.


## 1. Setup and data loading

The main emissions dataset is the Global Carbon Budget country-level dataset. Additional socioeconomic and energy variables are loaded from Our World in Data (OWID).


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import country_converter as coco

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [ ]:
# Main emissions dataset (included in data/raw/)
df = pd.read_csv('../data/raw/GCB2022v27_MtCO2_flat.csv')
df_1925 = df[df['Year'] >= 1925].copy()
df_1925 = df_1925.drop(columns=['Other'])

print(f"Emissions data shape (1925+): {df_1925.shape}")
df_1925.head()


### Initial exploration

The analysis starts by checking the time coverage, largest emitters, and missing-value patterns before adding external predictors.


In [ ]:
top10 = (df[df['Country'] != 'Global']
         .groupby('Country')['Total'].sum()
         .sort_values(ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
top10.plot(kind='bar', ax=axes[0], title='Top 10 countries by cumulative CO₂ emissions')
df.groupby('Year')['Total'].sum().plot(ax=axes[1], title='Total CO₂ emissions over time')
axes[0].set_ylabel('MtCO₂')
axes[1].set_ylabel('MtCO₂')
plt.tight_layout()
plt.show()

missing = pd.DataFrame({
    'missing_count': df_1925.isnull().sum(),
    'missing_pct': (df_1925.isnull().sum() / len(df_1925) * 100).round(2)
})
missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)


## 2. Enriching the dataset

OWID variables are merged by country and year. Non-country aggregate/historical entities are removed, and a continent feature is added for grouped imputations.


In [ ]:
owid = pd.read_csv('https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv')
extra_cols = [
    'country', 'year', 'gdp', 'population', 'energy_per_capita',
    'primary_energy_consumption', 'co2_per_gdp', 'methane', 'nitrous_oxide'
]
owid_subset = owid[extra_cols]

df_1925['Country'] = df_1925['Country'].replace({
    'Czech Republic': 'Czechia',
    'Viet Nam': 'Vietnam'
})

df_merged = df_1925.merge(
    owid_subset,
    left_on=['Country', 'Year'],
    right_on=['country', 'year'],
    how='left'
).drop(columns=['country', 'year'])

non_country_entities = [
    'Global', 'International Transport', 'Kuwaiti Oil Fires',
    'French Equatorial Africa', 'French West Africa', 'Leeward Islands',
    'Ryukyu Islands', 'St. Kitts-Nevis-Anguilla', 'Panama Canal Zone',
    'Anguilla', 'Antarctica', 'Bermuda', 'Bonaire, Saint Eustatius and Saba',
    'British Virgin Islands', 'Christmas Island', 'Cook Islands', 'Curaçao',
    'Faeroe Islands', 'French Guiana', 'French Polynesia', 'Greenland',
    'Guadeloupe', 'Hong Kong', 'Macao', 'Martinique', 'Mayotte', 'Montserrat',
    'New Caledonia', 'Niue', 'Puerto Rico', 'Réunion', 'Saint Helena',
    'Saint Pierre and Miquelon', 'Sint Maarten (Dutch part)', 'Taiwan',
    'Turks and Caicos Islands', 'Wallis and Futuna Islands'
]

df_clean = df_merged[~df_merged['Country'].isin(non_country_entities)].copy()

df_clean['Continent'] = coco.convert(
    names=df_clean['ISO 3166-1 alpha-3'].tolist(),
    to='continent', not_found='Unknown'
)
continent_map = {'Africa': 1, 'Asia': 2, 'Europe': 3, 'America': 4, 'Oceania': 5, 'Unknown': 0}
df_clean['Continent_encoded'] = df_clean['Continent'].map(continent_map)
df_clean.loc[df_clean['Country'] == 'Kosovo', ['Continent', 'Continent_encoded']] = ['Europe', 3]


## 3. Preliminary imputations and base dataset

A small set of variables is filled using domain-informed grouped statistics or deterministic relationships. Rows with unresolved continent mapping are removed. This creates the common starting point (`df_base`) for the four main missing-value strategies.


In [ ]:
df_clean['methane'] = df_clean.groupby(['Continent_encoded', 'Year'])['methane'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['population'] = df_clean.groupby('Continent_encoded')['population'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)
df_clean['nitrous_oxide'] = df_clean.groupby(['Continent_encoded', 'Year'])['nitrous_oxide'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['Per Capita'] = df_clean['Per Capita'].fillna(df_clean['Total'] / df_clean['population'])
df_clean['co2_per_gdp'] = df_clean['co2_per_gdp'].fillna(df_clean['Total'] / df_clean['gdp'])

df_clean = df_clean[df_clean['Continent_encoded'] != 0].copy()
df_clean = df_clean.drop(columns=['Continent', 'Country', 'ISO 3166-1 alpha-3'])
df_base = df_clean.copy()

print(f"Base dataset shape: {df_base.shape}")
print(df_base.isnull().sum()[df_base.isnull().sum() > 0])


## 4. Four missing-value strategies

- **V1 — deletion:** drop the two most incomplete energy columns and then remove remaining incomplete rows.
- **V2 — grouped median:** impute by continent and decade, with global medians as fallback.
- **V3 — missForest-style:** `IterativeImputer` with a `RandomForestRegressor` estimator.
- **V4 — MICE-style with temporal information:** `IterativeImputer` with `BayesianRidge`; `Year` remains an input feature and the data are sorted chronologically before imputation.

The V3 label is intentionally described as **missForest-style**, because this implementation uses scikit-learn's iterative imputer with a random-forest estimator rather than the original missForest package/algorithm.


In [ ]:
# V1 — deletion
#Korak 1: Drop kolona sa >50% missing (energy_per_capita, primary_energy_consumption)
cols_to_drop = ['energy_per_capita', 'primary_energy_consumption']
df_variant1 = df_base.drop(columns=cols_to_drop)


df_variant1 = df_variant1.dropna()

print(f"df_base shape:     {df_base.shape}")
print(f"df_variant1 shape: {df_variant1.shape}")
print(f"Izbačenih redova:  {len(df_base) - len(df_variant1)}")
print(f"\nMissing vrednosti u df_variant1:")
print(df_variant1.isnull().sum())


In [ ]:
# V2 — grouped median


df_variant2 = df_base.copy()

# Pomoćna kolona dekade
df_variant2['decade'] = (df_variant2['Year'] // 10) * 10

for col in ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring']:
    df_variant2[col] = df_variant2.groupby(['Continent_encoded', 'decade'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    df_variant2[col] = df_variant2[col].fillna(df_variant2[col].median())


df_variant2['gdp'] = df_variant2.groupby(['Continent_encoded', 'decade'])['gdp'].transform(
    lambda x: x.fillna(x.median())
)
df_variant2['gdp'] = df_variant2['gdp'].fillna(df_variant2['gdp'].median())


df_variant2['co2_per_gdp'] = df_variant2['co2_per_gdp'].fillna(
    df_variant2['Total'] / df_variant2['gdp']
)
df_variant2['co2_per_gdp'] = df_variant2.groupby(['Continent_encoded', 'decade'])['co2_per_gdp'].transform(
    lambda x: x.fillna(x.median())
)
df_variant2['co2_per_gdp'] = df_variant2['co2_per_gdp'].fillna(df_variant2['co2_per_gdp'].median())


for col in ['energy_per_capita', 'primary_energy_consumption']:
    df_variant2[col] = df_variant2.groupby(['Continent_encoded', 'decade'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    df_variant2[col] = df_variant2[col].fillna(df_variant2[col].median())

df_variant2 = df_variant2.drop(columns=['decade'])

print(f"df_base shape:     {df_base.shape}")
print(f"df_variant2 shape: {df_variant2.shape}")
print(f"\nMissing vrednosti u df_variant2:")
print(df_variant2.isnull().sum())


In [ ]:
# V3 — missForest-style

# IterativeImputer + RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

df_variant3 = df_base.select_dtypes(include='number').copy()

imputer_v3 = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    max_iter=10,
    random_state=42
)

imputed = imputer_v3.fit_transform(df_variant3)
df_variant3 = pd.DataFrame(imputed, columns=df_variant3.columns, index=df_variant3.index)

clip_cols = ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring',
             'energy_per_capita', 'primary_energy_consumption', 'gdp', 'co2_per_gdp']
df_variant3[clip_cols] = df_variant3[clip_cols].clip(lower=0)

print(f"df_base shape:     {df_base.shape}")
print(f"df_variant3 shape: {df_variant3.shape}")
print(f"\nMissing vrednosti u df_variant3:")
print(df_variant3.isnull().sum())


In [ ]:
# V4 — MICE-style with temporal information
from sklearn.linear_model import BayesianRidge

df_variant4 = df_base.select_dtypes(include='number').copy()
df_variant4 = df_variant4.sort_values('Year').reset_index(drop=True)

imputer_v4 = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42,
    initial_strategy='median'
)

imputed = imputer_v4.fit_transform(df_variant4)
df_variant4 = pd.DataFrame(imputed, columns=df_variant4.columns)

clip_cols = ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring',
             'energy_per_capita', 'primary_energy_consumption', 'gdp', 'co2_per_gdp']
df_variant4[clip_cols] = df_variant4[clip_cols].clip(lower=0)

print(f"df_base shape:     {df_base.shape}")
print(f"df_variant4 shape: {df_variant4.shape}")
print(f"\nMissing vrednosti u df_variant4:")
print(df_variant4.isnull().sum())


## 5. Leakage check and feature selection

An early random-split experiment produced near-perfect results. Investigation showed that several predictors directly define or derive from the target: `Total` is composed from fuel/process emission components, while `Per Capita` and `co2_per_gdp` are target-derived ratios. Keeping these variables would let the model effectively see the answer.

They are therefore excluded from the predictive feature set:
`Coal`, `Oil`, `Gas`, `Cement`, `Flaring`, `Per Capita`, and `co2_per_gdp`.


In [ ]:
LEAK_COLS = ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring', 'Per Capita', 'co2_per_gdp']

for name, frame in {
    'V1 (Deletion)': df_variant1,
    'V2 (Median)': df_variant2,
    'V3 (missForest-style)': df_variant3,
    'V4 (MICE-style)': df_variant4,
}.items():
    numeric = frame.select_dtypes(include='number').dropna()
    features = numeric.drop(columns=['Total'] + LEAK_COLS)
    print(f"{name}: {features.shape[1]} predictive features, {len(features)} rows")


## 6. Rolling-window validation

Because the observations are time-indexed, the final evaluation uses rolling windows rather than a random train/test split. Each fold trains on 20 years, tests on the following 5 years, and advances by 5 years.

**Important methodological limitation:** V1 and V2 can be fit using training-fold information only. V3 and V4 were pre-imputed on the full dataset before rolling-window evaluation because repeatedly fitting the iterative imputers for every fold was computationally expensive. This can introduce temporal leakage in the imputation step, so V3/V4 results should be interpreted with caution rather than as leakage-free production estimates.


In [ ]:
# ── ĆELIJA 2: Imputation helpers + rolling window CV ─────────────────────────
import numpy as np
import pandas as pd
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')

# ── V1: brisanje kolona i redova ──────────────────────────────────────────────
def impute_v1(train_df, test_df):
    drop = ['energy_per_capita', 'primary_energy_consumption']
    train = train_df.drop(columns=[c for c in drop if c in train_df.columns]).dropna()
    test  = test_df.drop(columns=[c for c in drop  if c in test_df.columns]).dropna()
    return train, test

# ── V2: medijana računata SAMO na trening delu — bez curenja u test ───────────
def impute_v2(train_df, test_df):
    fill_cols = ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring', 'gdp',
                 'co2_per_gdp', 'energy_per_capita', 'primary_energy_consumption']
    fill_cols = [c for c in fill_cols if c in train_df.columns]
    train = train_df.copy()
    test  = test_df.copy()
    for col in fill_cols:
        med = train[col].median()           # medijana SAMO iz trening skupa
        train[col] = train[col].fillna(med)
        test[col]  = test[col].fillna(med)  # primenjuje se na test (bez curenja)
    return train, test

# ── Rolling window CV ─────────────────────────────────────────────────────────
LEAK_COLS  = ['Coal', 'Oil', 'Gas', 'Cement', 'Flaring', 'Per Capita', 'co2_per_gdp']
CV_START   = 1960   # početna godina (gde je gustina ≥100 zemalja za sve varijante)
TRAIN_YRS  = 20     # širina trening prozora
TEST_YRS   = 5      # širina test prozora
STEP       = 5      # pomak prozora po foldu

def rolling_window_cv(df_source, impute_fn, model_factory,
                      needs_scaling=False,
                      cv_start=CV_START, train_yrs=TRAIN_YRS,
                      test_yrs=TEST_YRS, step=STEP):
    """
    Klizeći prozor: trening [yr, yr+train_yrs-1], test [yr+train_yrs, yr+train_yrs+test_yrs-1].
    Prozor se pomera za `step` godina. Sve države čija je Year u opsegu idu u isti skup.
    """
    max_year = df_source['Year'].max()
    results  = []
    yr       = cv_start

    while True:
        t_start = yr
        t_end   = yr + train_yrs - 1
        s_start = t_end + 1
        s_end   = s_start + test_yrs - 1

        if s_end > max_year:
            break

        train_df = df_source[(df_source['Year'] >= t_start) &
                             (df_source['Year'] <= t_end)].copy()
        test_df  = df_source[(df_source['Year'] >= s_start) &
                             (df_source['Year'] <= s_end)].copy()

        # Imputacija (None za V3/V4 — pre-imputirani ceo skup, videti napomenu)
        if impute_fn is not None:
            train_df, test_df = impute_fn(train_df, test_df)

        # Priprema feature matrica
        drop_cols = ['Total'] + [c for c in LEAK_COLS if c in train_df.columns]

        tr_num = train_df.select_dtypes(include='number').dropna()
        te_num = test_df.select_dtypes(include='number').dropna()

        X_tr = tr_num.drop(columns=[c for c in drop_cols if c in tr_num.columns])
        y_tr = tr_num['Total']
        X_te = te_num.drop(columns=[c for c in drop_cols if c in te_num.columns])
        y_te = te_num['Total']

        # Uskladi kolone (V1 ima 6, V2-V4 imaju 8)
        cols = sorted(set(X_tr.columns) & set(X_te.columns))
        X_tr = X_tr[cols].values
        X_te = X_te[cols].values

        if len(X_tr) < 20 or len(X_te) < 10:
            yr += step
            continue

        if needs_scaling:
            sc   = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

        model = model_factory()
        model.fit(X_tr, y_tr.values)
        y_pred = model.predict(X_te)

        results.append({
            'fold':       f"{t_start}–{t_end} → {s_start}–{s_end}",
            'train_rows': len(X_tr),
            'test_rows':  len(X_te),
            'MAE':        mean_absolute_error(y_te, y_pred),
            'RMSE':       np.sqrt(mean_squared_error(y_te, y_pred)),
            'R²':         r2_score(y_te, y_pred),
        })
        yr += step

    return pd.DataFrame(results)


## 7. Models

Four model families are compared: linear regression, a scikit-learn multilayer perceptron, random forest, and a custom PyTorch Transformer-style model for tabular numeric features.


In [ ]:
# ── ĆELIJA 3: sklearn-kompatibilni TabTransformer wrapper ────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class _TabNet(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(1, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, 128, 0.1, batch_first=True)
        self.tr  = nn.TransformerEncoder(enc, num_layers)
        self.out = nn.Sequential(
            nn.Linear(d_model * input_dim, 64), nn.ReLU(), nn.Linear(64, 1)
        )
    def forward(self, x):
        x = self.proj(x.unsqueeze(-1))
        x = self.tr(x)
        return self.out(x.reshape(x.size(0), -1)).squeeze(-1)

class TabTransformerCV:
    """Wrapper koji se ponaša kao sklearn estimator (fit/predict)."""
    def __init__(self, epochs=100, batch_size=64, lr=1e-3):
        self.epochs, self.batch_size, self.lr = epochs, batch_size, lr
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.net_   = None

    def fit(self, X, y):
        self.net_ = _TabNet(X.shape[1]).to(self.device)
        opt  = torch.optim.Adam(self.net_.parameters(), lr=self.lr)
        loss_fn = nn.MSELoss()
        Xt = torch.tensor(X, dtype=torch.float32).to(self.device)
        yt = torch.tensor(y, dtype=torch.float32).to(self.device)
        loader = DataLoader(TensorDataset(Xt, yt), self.batch_size, shuffle=True)
        self.net_.train()
        for _ in range(self.epochs):
            for xb, yb in loader:
                opt.zero_grad()
                loss_fn(self.net_(xb), yb).backward()
                opt.step()
        return self

    def predict(self, X):
        Xt = torch.tensor(X, dtype=torch.float32).to(self.device)
        self.net_.eval()
        with torch.no_grad():
            return self.net_(Xt).cpu().numpy()


In [ ]:
import time

variants_cv = {
    'V1 (Deletion)':          (df_base,     impute_v1),
    'V2 (Median)':            (df_base,     impute_v2),
    'V3 (missForest-style)':  (df_variant3, None),
    'V4 (MICE-style)':        (df_variant4, None),
}

models_cv = {
    'Linear Regression': (lambda: LinearRegression(), False),
    'MLP': (lambda: MLPRegressor(
        hidden_layer_sizes=(64, 32), activation='relu', max_iter=500,
        random_state=RANDOM_STATE, early_stopping=True, validation_fraction=0.1), True),
    'Random Forest': (lambda: RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1), False),
    'Transformer': (lambda: TabTransformerCV(epochs=100, batch_size=64, lr=1e-3), True),
}

# Full execution is computationally intensive, especially the Transformer.
# Set RUN_FULL_EXPERIMENT = True to reproduce all rolling-window results.
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    all_results = {}
    for variant_name, (source_df, imputer_fn) in variants_cv.items():
        for model_name, (model_factory, scaling) in models_cv.items():
            start = time.time()
            fold_df = rolling_window_cv(
                df_source=source_df,
                impute_fn=imputer_fn,
                model_factory=model_factory,
                needs_scaling=scaling,
            )
            all_results[(variant_name, model_name)] = fold_df
            print(f"{variant_name} × {model_name}: {len(fold_df)} folds ({time.time()-start:.1f}s)")


## 8. Saved experiment results

The repository includes the results from the completed 8-fold experiment so the findings can be inspected without rerunning the computationally expensive Transformer training.


In [ ]:
results = pd.read_csv('../results/cv_results.csv')
results.sort_values(['Model', 'R²_mean'], ascending=[True, False])


In [ ]:
# Best imputation strategy within each model according to mean R²
results.loc[results.groupby('Model')['R²_mean'].idxmax(),
            ['Model', 'Varijanta', 'MAE_mean', 'RMSE_mean', 'R²_mean']]


## 9. Key findings

Across the saved rolling-window results, the MICE-style variant produced the highest mean R² for all four model families. For example, it reached mean R² values of **0.9885** with linear regression, **0.9918** with the MLP, **0.9790** with random forest, and **0.9669** with the Transformer.

The experiment also illustrates two practical lessons: target-derived features can create deceptively perfect scores, and validation design matters for time-indexed data. The V3/V4 pre-imputation limitation means the reported comparison is best treated as an academic experiment and a basis for further leakage-safe evaluation.
